In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
import os
import cstarpy.integration
COMPOUND_FILE = "/Users/samuelnugent/Desktop/Masters/Spring/MEIN40430/Thesis/New Results/data/cd8_limma_merged_filtered_targets_ic50_dpd.csv"
SELECTION     = "top4_bottom4_selected_modules_drugs_ic50.csv"   # from prep notebook
out_dir       = "02_outputs"
os.makedirs(out_dir, exist_ok=True)

# Load the collapsed-module selection built in prep (module column already exists)
drug_gene = pd.read_csv(SELECTION).rename(columns={"compound_name": "drug"})

modules  = drug_gene["module"].drop_duplicates().tolist()
exp_list = drug_gene["drug"].drop_duplicates().tolist()

print(f"Modules ({len(modules)}): {modules}")
print(f"Experiments ({len(exp_list)}): {exp_list}")

Modules (7): ['ARFGAP', 'BCL', 'IGF1R', 'JAK', 'MTOR', 'NAE', 'SERCA']
Experiments (12): ['QS-11', 'navitoclax', 'BMS-536924', 'AT9283', 'TG-101348', 'CYT-387', 'ruxolitinib', 'Deforolimus', 'Temsirolimus', 'Sapanisertib', 'Pevonedistat', 'Thapsigargin']


In [2]:
df = pd.read_csv(COMPOUND_FILE)

df_avg = (
    df.groupby(["compound_name", "gene"])
    .agg(logFC=("logFC", "mean"), adj_P_Val=("adj.P.Val", "mean"))
    .reset_index()
)
df_avg["logFC_thresh"] = np.where(df_avg["adj_P_Val"] < 0.05, df_avg["logFC"], 0)

x_df = df_avg.pivot_table(
    index="gene", columns="compound_name", values="logFC_thresh", aggfunc="first"
).fillna(0)[exp_list]

genes = x_df.index.tolist()
x     = x_df.values
print(f"x (expression) shape: {x.shape}  (genes × experiments)")

x (expression) shape: (15045, 12)  (genes × experiments)


In [3]:
dose_info = df.groupby("compound_name")["dose_uM"].first()
mech_info = df.groupby("compound_name")["mechanism"].first()

inhib_conc_matrix_top4_bottom4 = np.zeros((len(modules), len(exp_list)))
ic50_matrix_top4_bottom4       = np.ones((len(modules), len(exp_list))) * np.inf   # inf → g=1
gamma_matrix_top4_bottom4      = np.zeros((len(modules), len(exp_list)))            # 0 = inhibitor, >0 = activator
valid_matrix_top4_bottom4      = np.zeros((len(modules), len(exp_list)), dtype=bool)  # has real dose+IC50 data

GAMMA = 1.0   # activation coefficient, applied only where mechanism == "Activator"

for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        dose = dose_info.get(row["drug"], np.nan)
        ic50_raw = df[df["compound_name"] == row["drug"]]["IC50_nM"]
        ic50_nm = ic50_raw.iloc[0] if len(ic50_raw) else np.nan
        ic50 = ic50_nm / 1000 if pd.notna(ic50_nm) else np.nan
        mech = mech_info.get(row["drug"], "Inhibitor")
        if pd.notna(dose) and pd.notna(ic50):
            inhib_conc_matrix_top4_bottom4[i, j] = dose
            ic50_matrix_top4_bottom4[i, j]       = ic50
            gamma_matrix_top4_bottom4[i, j]      = GAMMA if mech == "Activator" else 0.0
            valid_matrix_top4_bottom4[i, j]      = True
        else:
            print(f"WARNING: missing dose/IC50 for module={row['module']} drug={row['drug']} "
                  f"— excluded from fit (not treated as 'no effect')")

# y_true = (1 + gamma_matrix * dose/IC50) / (1 + dose/IC50)   [gamma=0 collapses to inhibitor form]
dratio_top4_bottom4 = inhib_conc_matrix_top4_bottom4 / ic50_matrix_top4_bottom4
y_true_top4_bottom4 = np.where(
    valid_matrix_top4_bottom4,
    (1 + gamma_matrix_top4_bottom4 * dratio_top4_bottom4) / (1 + dratio_top4_bottom4),
    1.0,
)

print(f"y_true (activity g) shape: {y_true_top4_bottom4.shape}")
print(pd.DataFrame(y_true_top4_bottom4, index=modules, columns=exp_list).round(3).to_string())


y_true (activity g) shape: (7, 12)
        QS-11  navitoclax  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Deforolimus  Temsirolimus  Sapanisertib  Pevonedistat  Thapsigargin
ARFGAP   0.13       1.000       1.000   1.000      1.000    1.000          1.0        1.000           1.0           1.0         1.000         1.000
BCL      1.00       0.002       1.000   1.000      1.000    1.000          1.0        1.000           1.0           1.0         1.000         1.000
IGF1R    1.00       1.000       0.079   1.000      1.000    1.000          1.0        1.000           1.0           1.0         1.000         1.000
JAK      1.00       1.000       1.000   0.038      0.041    0.099          0.0        1.000           1.0           1.0         1.000         1.000
MTOR     1.00       1.000       1.000   1.000      1.000    1.000          1.0        0.002           0.0           0.0         1.000         1.000
NAE      1.00       1.000       1.000   1.000      1.000    1.000          1.

In [4]:
pert_matrix_top4_bottom4 = np.zeros((len(modules), len(exp_list)))
for _, row in drug_gene.iterrows():
    if row["module"] in modules and row["drug"] in exp_list:
        i = modules.index(row["module"])
        j = exp_list.index(row["drug"])
        pert_matrix_top4_bottom4[i, j] = 1

fit_mask_top4_bottom4 = pert_matrix_top4_bottom4 * valid_matrix_top4_bottom4
print(f"pert_matrix shape: {pert_matrix_top4_bottom4.shape}")
print(pd.DataFrame(pert_matrix_top4_bottom4.astype(int), index=modules, columns=exp_list).to_string())
print(f"\nfit_mask (excludes missing-IC50 pairs):")
print(pd.DataFrame(fit_mask_top4_bottom4.astype(int), index=modules, columns=exp_list).to_string())


pert_matrix shape: (7, 12)
        QS-11  navitoclax  BMS-536924  AT9283  TG-101348  CYT-387  ruxolitinib  Deforolimus  Temsirolimus  Sapanisertib  Pevonedistat  Thapsigargin
ARFGAP      1           0           0       0          0        0            0            0             0             0             0             0
BCL         0           1           0       0          0        0            0            0             0             0             0             0
IGF1R       0           0           1       0          0        0            0            0             0             0             0             0
JAK         0           0           0       1          1        1            1            0             0             0             0             0
MTOR        0           0           0       0          0        0            0            1             1             1             0             0
NAE         0           0           0       0          0        0            0       

In [5]:
residuals, a_coeffs = cstarpy.integration.pathway_activity.prediction.predict_coeffs(
    x, y_true_top4_bottom4, fit_mask_top4_bottom4,
    200_000, 10, 10, 10, 100
)

a_coeffs_df_top4_bottom4 = pd.DataFrame(a_coeffs, index=modules, columns=genes)
a_coeffs_df_top4_bottom4.to_csv(os.path.join(out_dir, "a_coeffs_df_top4_bottom4.csv"))
print(f"a_coeffs shape: {a_coeffs.shape}")
trh = 0.0001


100%|██████████| 200000/200000 [03:14<00:00, 1027.30it/s]


a_coeffs shape: (7, 15045)


In [24]:
print("\nGenes representing each module:")
print((abs(a_coeffs_df_top4_bottom4) > trh).sum(axis="columns").to_string())


Genes representing each module:
ARFGAP     4
BCL        1
IGF1R      4
JAK       28
MTOR       5
NAE        4
SERCA      4


In [6]:
# Use .values for matrix math (a_coeffs as array, not DataFrame)
a_coeffs_top4_bottom4 = a_coeffs_df_top4_bottom4.values

pathway_activity_top4_bottom4 = a_coeffs_top4_bottom4 @ x
pd.DataFrame(pathway_activity_top4_bottom4, index=modules, columns=exp_list)\
    .to_csv(os.path.join(out_dir, "pathway_activity_top4_bottom4.csv"))

R_global_top4_bottom4 = cstarpy.integration.pathway_activity.calc_global_response_from_pathway_activity(
    cstarpy.integration.pathway_activity.calc_pathway_activity(x, a_coeffs_top4_bottom4),
    modules, exp_list
)
R_global_df_top4_bottom4 = pd.DataFrame(R_global_top4_bottom4, index=modules, columns=exp_list)
R_global_df_top4_bottom4.to_csv(os.path.join(out_dir, "R_global_core_top4_bottom4.csv"))

pd.DataFrame(y_true_top4_bottom4,      index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "y_true_top4_bottom4.csv"))
pd.DataFrame(pert_matrix_top4_bottom4, index=modules, columns=exp_list).to_csv(os.path.join(out_dir, "pert_matrix_top4_bottom4.csv"))
x_df.to_csv(os.path.join(out_dir, "Data_norm.csv"))


In [7]:
R_global_df_top4_bottom4

,QS-11,navitoclax,BMS-536924,AT9283,TG-101348,CYT-387,ruxolitinib,Deforolimus,Temsirolimus,Sapanisertib,Pevonedistat,Thapsigargin
ARFGAP,-1.251897,0.000055,-0.002044,-0.000051,-0.000934,-0.001315,-0.001723,0.184803,-0.001965,-0.000768,-0.001874,-0.331116
BCL,0.000651,-1.595838,0.000412,0.000050,-0.000123,0.000869,0.002239,0.000577,0.000857,0.001344,0.000567,-0.748127
IGF1R,-0.001197,0.000001,-1.434390,-0.000065,-0.002515,-0.002695,-0.008419,-0.000128,-0.002213,-0.005360,-0.000954,-0.005362
JAK,-1.220107,0.002311,-1.946056,-1.146188,-1.530240,-1.347160,-1.924917,-0.011590,-1.183935,-1.919870,-0.641175,-1.980169
MTOR,-0.000915,0.000094,-0.003223,-0.000062,-0.001078,-0.002303,-0.008697,-1.631300,-1.917932,-1.971792,-0.000503,-0.003579
NAE,-0.756852,-0.000083,-0.001483,-0.000017,-0.000469,-0.000653,-0.002772,-0.001519,-0.001853,-0.002095,-1.410626,-1.912829
SERCA,-0.352287,0.000254,-0.002348,-0.000019,-0.463551,-0.001847,-0.007065,-0.002219,-0.001398,-0.011064,-0.327343,-1.767545


In [8]:
pert_matrix_top4_bottom4

array([[1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 1., 1., 1., 1., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 1., 1., 1., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1.]])

In [21]:
a_coeffs_df_top4_bottom4

,A1BG,A2M,A2M-AS1,A2MP1,A4GALT,AAAS,AACS,AAGAB,AAK1,AAMDC,...,ZSWIM8,ZSWIM9,ZUP1,ZW10,ZWILCH,ZWINT,ZXDC,ZYG11B,ZYX,ZZEF1
ARFGAP,0.000014,1.364390e-05,0.000008,1.474209e-05,-7.889425e-06,-0.000018,-1.085749e-05,0.000023,0.000005,-2.386848e-05,...,0.000011,-0.000009,-0.000012,-8.239865e-06,1.248963e-05,-0.000006,-0.000003,1.977724e-05,-1.192358e-05,0.000012
BCL,0.000010,-1.250738e-05,0.000001,-2.234727e-06,-6.679264e-06,-0.000003,3.351969e-06,-0.000021,0.000004,-1.212600e-05,...,-0.000008,-0.000012,0.000006,2.829072e-06,-1.361564e-05,-0.000010,-0.000020,7.563080e-07,-2.434062e-07,0.000006
IGF1R,-0.000012,-1.003070e-05,-0.000023,1.420473e-06,-2.255321e-05,0.000015,-6.289081e-06,-0.000006,0.000011,-7.347116e-06,...,-0.000015,0.000008,0.000006,-3.347543e-07,1.134990e-05,0.000005,-0.000002,1.389572e-05,-1.007402e-06,0.000017
JAK,-0.000016,1.923837e-07,-0.000004,6.970738e-08,2.886512e-05,0.000013,1.387631e-05,-0.000005,-0.000018,4.495577e-07,...,-0.000013,-0.000040,-0.000008,-2.894332e-06,-1.734679e-05,-0.000017,0.000009,-1.087360e-05,-3.009062e-05,-0.000002
MTOR,-0.000014,-5.108792e-06,0.000003,9.246236e-06,-6.978064e-06,0.000023,2.429650e-05,-0.000017,-0.000008,-1.221587e-06,...,0.000002,-0.000010,-0.000014,1.289835e-05,-7.429599e-07,0.000003,0.000017,1.669737e-05,1.379419e-05,0.000001
NAE,-0.000018,-1.359107e-05,-0.000015,5.767643e-06,-8.196423e-07,-0.000008,5.856262e-08,-0.000004,-0.000022,-6.284023e-06,...,-0.000007,0.000011,0.000012,-1.661298e-06,-4.557327e-06,-0.000012,0.000009,1.175694e-05,-4.670086e-05,0.000006
SERCA,-0.000009,2.925955e-05,-0.000013,-4.564622e-06,-1.921576e-07,-0.000009,9.644061e-06,-0.000016,-0.000002,-1.157314e-05,...,0.000006,0.000014,0.000016,-8.738381e-06,8.170177e-06,0.000023,0.000003,6.726724e-06,5.524450e-06,-0.000002


In [22]:
residuals

array([6.7552922e+04, 6.6088898e+04, 6.4664336e+04, ..., 4.6435513e+01,
       4.7152435e+01, 4.6642845e+01], shape=(200000,), dtype=float32)